# Predict the next token with a tiny GPT

A GPT is trained to do one thing: predict the next token from the tokens before it. Here you
build that in NumPy on eight toy sentences: the chain rule that turns next-token probabilities
into the probability of a sentence, one causal self-attention layer, the shifted targets and
their loss, and an output layer trained for a few hundred steps. At the end the model writes
sentences on its own, greedily and by sampling at several temperatures.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

sentences = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat saw the big dog",
    "the dog saw the big cat",
    "a cat ate the red fish",
    "a dog ate the red bone",
    "the cat ran to the mat",
    "the dog ran to the rug",
]
specials = ["<pad>", "<s>", "<e>"]
vocab = specials + sorted({w for s in sentences for w in s.split()})
idx = {w: i for i, w in enumerate(vocab)}
PAD, START, END = 0, 1, 2
n_vocab = len(vocab)


def to_ids(text):
    """Token ids with <s> in front and <e> at the end."""
    return np.array([START] + [idx[w] for w in text.split()] + [END])


def to_words(ids):
    return " ".join(vocab[i] for i in ids)


def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


tokens = np.stack([to_ids(s) for s in sentences])   # 8 sequences: <s>, 6 words, <e>
T = tokens.shape[1]                                  # tokens per sequence
d = 16                                               # width of every vector inside the model

# Random starting weights. Only W_out and b_out will be trained.
E = 0.5 * rng.normal(size=(n_vocab, d))             # token embeddings
P = 0.5 * rng.normal(size=(T, d))                   # position embeddings, one per position
W_q, W_k, W_v = (rng.normal(size=(d, d)) / np.sqrt(d) for _ in range(3))
W_out = rng.normal(size=(d, n_vocab)) / np.sqrt(d)  # output layer: one score per vocabulary entry
b_out = np.zeros(n_vocab)


def embed(ids):
    """One vector per token: its token embedding plus the embedding of its position. Shape (len(ids), d)."""
    return E[ids] + P[np.arange(len(ids))]


print(n_vocab, "entries in the vocabulary:", vocab)
print("sequence 0:", tokens[0], "->", to_words(tokens[0]))

## Part 1 (coding task): The probability of a sentence

A language model gives each token a probability given the tokens before it. The chain
rule of probability multiplies those together: the probability of "the cat sat" is
`P(the) * P(cat | the) * P(sat | the cat)`. The loss is the average of `-ln` of the same
numbers, and the perplexity is `e` raised to the loss. A perplexity of `k` means the model
was, on average, as unsure as if it were choosing evenly among `k` tokens.

Write `score_sentence(conditionals)`. It takes a list of the conditional probabilities,
one per token in order, and returns `(probability, loss, perplexity)`. The lesson's
numbers are 0.2, 0.1 and 0.5.

In [ ]:
def score_sentence(conditionals):
    """(probability of the whole sentence, mean loss per token, perplexity)."""
    # TODO
    raise NotImplementedError

In [ ]:
prob, loss, ppl = score_sentence([0.2, 0.1, 0.5])
assert abs(prob - 0.01) < 1e-9, f"0.2 * 0.1 * 0.5 = 0.01, got {prob}"
assert abs(loss - 1.5351) < 1e-3, f"the mean of -ln should be 1.535, got {loss:.4f}"
assert abs(ppl - 4.642) < 1e-2, f"e^1.535 = 4.64, got {ppl:.3f}"
for k in (2, 10, n_vocab):
    _, _, ppl_k = score_sentence([1 / k] * 5)
    assert abs(ppl_k - k) < 1e-9, f"evenly spread over {k} tokens means perplexity {k}, got {ppl_k}"
_, loss_sure, _ = score_sentence([1.0, 1.0, 1.0])
assert abs(loss_sure) < 1e-12, "a model that is sure of every token has loss 0"
print(f"P(the cat sat) = {prob:.3f}, loss {loss:.3f}, perplexity {ppl:.2f}")
print("Part 1 passed")

## Part 2 (coding task): Causal self-attention

Attention scores form a `T x T` matrix: row `t` is the query at position `t`, and column
`s` is the key at position `s`. Entries above the diagonal have `s > t`, so they point to
the future. A GPT sets those scores to minus infinity before the softmax, and
`exp(-inf) = 0` turns them into weights of exactly 0. That is the causal mask. With it,
the output at position `t` depends only on tokens up to `t`, so one pass over a sequence
gives a prediction at every position at once. `np.tril(np.ones((n, n), dtype=bool))`
marks the allowed entries, the diagonal and below.

Write `causal_self_attention(X)`. Compute `Q`, `K` and `V` with `W_q`, `W_k` and `W_v`,
the scores `Q @ K.T / np.sqrt(d)`, set the entries above the diagonal to `-np.inf`, take
the softmax along each row, and return `(X + weights @ V, weights)`. Use `len(X)` for the
size of the mask, because during generation the input is shorter than `T`.

In [ ]:
def causal_self_attention(X):
    """One layer of attention where each position sees only itself and earlier positions.
    Returns (H, weights): H is X plus what each position gathered, weights is the masked attention matrix."""
    # TODO
    raise NotImplementedError

In [ ]:
X = embed(tokens[0])
H, A = causal_self_attention(X)
assert H.shape == (T, d) and A.shape == (T, T), f"shapes should be {(T, d)} and {(T, T)}, got {H.shape} and {A.shape}"
assert np.allclose(A.sum(axis=1), 1), "each row of weights should sum to 1"
assert np.allclose(np.triu(A, 1), 0), "every weight above the diagonal should be exactly 0"
assert (A[np.tril(np.ones((T, T), dtype=bool))] > 0).all(), "the diagonal and everything below it should be positive"
assert np.isclose(A[0, 0], 1), "position 0 can see only itself"
Q, K, V = X @ W_q, X @ W_k, X @ W_v
S = Q @ K.T / np.sqrt(d)
S[np.triu(np.ones((T, T), dtype=bool), 1)] = -np.inf
A_ref = softmax(S, axis=1)
assert np.allclose(A, A_ref), "weights should be the row softmax of the masked Q K^T / sqrt(d)"
assert np.allclose(H, X + A_ref @ V), "H should be X plus weights @ V"
# Change the last token: the earlier outputs must not move at all.
changed = tokens[0].copy()
changed[-1] = idx["rug"]
H_changed, _ = causal_self_attention(embed(changed))
assert np.allclose(H_changed[:-1], H[:-1]), "a later token must not change the output at an earlier position"
assert not np.allclose(H_changed[-1], H[-1]), "the output at the changed position should move"
# A prefix on its own gives the same outputs as the first rows of the full sequence.
H_prefix, _ = causal_self_attention(embed(tokens[0][:3]))
assert np.allclose(H_prefix, H[:3]), "running the first 3 tokens alone should match the first 3 rows"
print(np.round(A, 3))
print("Part 2 passed")

## Part 3 (coding task): Shifted targets and the loss

The model reads tokens `0` to `T-2` and is scored on predicting tokens `1` to `T-1`. So
`inputs = tokens[:-1]` and `targets = tokens[1:]`. The output layer turns the vector at
each input position into one score per vocabulary entry, and the loss at a position is
`-ln` of the softmax probability of its target. Score position `t` against token `t`
instead of token `t+1` and the model only has to copy what it can see, so the loss drops
to almost zero and nothing is learned.

Write `lm_loss(tokens, W_out, b_out)` for one sequence. Shift, embed the inputs, run
`causal_self_attention`, compute `logits = H @ W_out + b_out`, and return the mean
cross-entropy against the targets. Compute the log of the softmax as `logits` minus the
row max minus the log of the sum of `exp(logits - row max)`.

In [ ]:
def lm_loss(tokens, W_out, b_out):
    """Mean next-token cross-entropy of one sequence: the model reads tokens[:-1] and is scored on tokens[1:]."""
    # TODO
    raise NotImplementedError

In [ ]:
zero_W, zero_b = np.zeros((d, n_vocab)), np.zeros(n_vocab)
loss0 = lm_loss(tokens[0], zero_W, zero_b)
assert abs(loss0 - np.log(n_vocab)) < 1e-9, f"all-equal scores should cost ln({n_vocab}) = {np.log(n_vocab):.3f}, got {loss0:.4f}"
# An output layer that is sure the next token is always "cat": free at one position, 60 at each of the other six.
sure_b = np.full(n_vocab, -30.0)
sure_b[idx["cat"]] = 30.0
loss_sure = lm_loss(tokens[0], zero_W, sure_b)
assert abs(loss_sure - 60 * 6 / 7) < 1e-6, f"expected {60 * 6 / 7:.4f}, got {loss_sure:.4f}"
# Against an independent computation on the untrained weights.
H, _ = causal_self_attention(embed(tokens[0][:-1]))
p = softmax(H @ W_out + b_out, axis=1)
expected = -np.mean(np.log(p[np.arange(T - 1), tokens[0][1:]]))
got = lm_loss(tokens[0], W_out, b_out)
assert abs(got - expected) < 1e-9, f"expected {expected:.4f}, got {got:.4f}: check the shift"
print(f"untrained loss on sequence 0: {got:.3f}, perplexity {np.exp(got):.1f}, vocabulary size {n_vocab}")
print("Part 3 passed")

## Part 4 (coding task): Train the output layer

Only the output layer learns here, which keeps the gradient short. For a softmax
classifier, the gradient of the mean loss with respect to the logits at a position is
`softmax(logits[t]) - onehot(targets[t])`, divided by the number of positions. Because
`logits = H @ W_out + b_out`, the gradient for `W_out` is `H.T @ dlogits` and the gradient
for `b_out` is the sum of the rows of `dlogits`.

Write `head_gradient(tokens, W_out, b_out)` for one sequence. It returns `(dW, db)` with
the shapes of `W_out` and `b_out`. The check compares it with finite differences, then
takes 300 gradient descent steps on the eight sequences, averaging the gradients over
them, and plots the loss.

In [ ]:
def head_gradient(tokens, W_out, b_out):
    """Gradient of lm_loss(tokens, W_out, b_out) with respect to W_out and b_out."""
    # TODO
    raise NotImplementedError

In [ ]:
dW, db = head_gradient(tokens[0], W_out, b_out)
assert dW.shape == W_out.shape and db.shape == b_out.shape, "return (dW, db) with the shapes of W_out and b_out"
eps = 1e-6
for i, j in [(0, 0), (7, 5), (15, n_vocab - 1)]:
    W_plus, W_minus = W_out.copy(), W_out.copy()
    W_plus[i, j] += eps
    W_minus[i, j] -= eps
    numeric = (lm_loss(tokens[0], W_plus, b_out) - lm_loss(tokens[0], W_minus, b_out)) / (2 * eps)
    assert abs(numeric - dW[i, j]) < 1e-5, f"dW[{i}, {j}] should be {numeric:.6f}, got {dW[i, j]:.6f}"
b_plus, b_minus = b_out.copy(), b_out.copy()
b_plus[3] += eps
b_minus[3] -= eps
numeric = (lm_loss(tokens[0], W_out, b_plus) - lm_loss(tokens[0], W_out, b_minus)) / (2 * eps)
assert abs(numeric - db[3]) < 1e-5, f"db[3] should be {numeric:.6f}, got {db[3]:.6f}"

W, b = W_out.copy(), b_out.copy()
losses = []
for step in range(300):
    losses.append(np.mean([lm_loss(seq, W, b) for seq in tokens]))
    grads = [head_gradient(seq, W, b) for seq in tokens]
    W -= 1.0 * np.mean([g[0] for g in grads], axis=0)
    b -= 1.0 * np.mean([g[1] for g in grads], axis=0)
plt.plot(losses)
plt.axhline(np.log(n_vocab), color="gray", linestyle="--", label="ln(n_vocab)")
plt.xlabel("step")
plt.ylabel("next-token loss")
plt.legend()
plt.show()
print(f"loss {losses[0]:.3f} -> {losses[-1]:.3f}, perplexity {np.exp(losses[0]):.1f} -> {np.exp(losses[-1]):.2f}")
assert losses[-1] < 1.0, f"after 300 steps the loss should be below 1, got {losses[-1]:.3f}"
# The trained model's conditional probabilities along sequence 0, scored with Part 1.
H, _ = causal_self_attention(embed(tokens[0][:-1]))
p = softmax(H @ W + b, axis=1)[np.arange(T - 1), tokens[0][1:]]
prob, loss, ppl = score_sentence(p)
print("P(next token) along '" + to_words(tokens[0]) + "':", np.round(p, 2))
print(f"probability of the whole sequence {prob:.4f}, loss {loss:.3f}, perplexity {ppl:.2f}")
assert abs(loss - lm_loss(tokens[0], W, b)) < 1e-9, "Part 1 and Part 3 should agree on the loss"
print("Part 4 passed")

## Part 5 (coding task): Greedy generation

To write, start from `<s>` and repeat: run the model on the tokens so far, take the
logits at the last position, pick a token, append it. Stop at `<e>`, or when the sequence
reaches `T` tokens, the number of position embeddings. Picking the most likely token
every time is greedy decoding. Because of the causal mask, the logits at the last position
of a prefix are the same ones the full sequence would give there, as Part 2 showed.

Write `generate_greedy(prompt_ids, W_out, b_out)`. It takes an array of ids that starts
with `<s>` and returns the full array of ids, prompt included, ending with `<e>` if the
model produced one.

In [ ]:
def generate_greedy(prompt_ids, W_out, b_out):
    """Extend prompt_ids one most-likely token at a time until <e> or T tokens. Returns all the ids."""
    # TODO
    raise NotImplementedError

In [ ]:
out = generate_greedy(np.array([START]), W, b)
assert out[0] == START, "the prompt stays at the front"
assert len(out) <= T, f"the model has {T} position embeddings, so it cannot write more than {T} tokens"
assert out[-1] == END or len(out) == T, "stop at <e> or when the sequence is full"
# Every token must be the model's most likely choice given the tokens before it.
for t in range(1, len(out)):
    H, _ = causal_self_attention(embed(out[:t]))
    assert out[t] == np.argmax(H[-1] @ W + b), f"token {t} is not the most likely choice after '{to_words(out[:t])}'"
print("greedy from <s>:", to_words(out))
for prompt in ["<s> a", "<s> the dog saw"]:
    prompt_ids = np.array([idx[w] for w in prompt.split()])
    print(f"greedy from '{prompt}':", to_words(generate_greedy(prompt_ids, W, b)))
assert to_words(out[1:-1]) in sentences, "greedy decoding from <s> should reproduce one of the training sentences"
done = to_ids("the cat sat on the mat")
assert (generate_greedy(done, W, b) == done).all(), "a prompt that already ends with <e> comes back unchanged"
print("Part 5 passed")

## Part 6 (coding task): Sampling and temperature

Greedy decoding writes the same text every time and often falls into a loop. Sampling
draws the next token from the model's probabilities instead. Temperature changes those
probabilities: divide every logit by `tau` before the softmax. Below 1 it sharpens the
distribution toward the top token, above 1 it flattens it, and at 1 nothing changes.

Write `sample_next(logits, tau, rng)`. It returns one id drawn with
`rng.choice(len(logits), p=probabilities)`, where the probabilities are the softmax of
`logits / tau`. The check also puts it inside a generation loop like Part 5's and prints
text at three temperatures.

In [ ]:
def sample_next(logits, tau, rng):
    """One id drawn from softmax(logits / tau)."""
    # TODO
    raise NotImplementedError

In [ ]:
check_rng = np.random.default_rng(0)
logits3 = np.log([0.6, 0.3, 0.1])
draws = np.array([sample_next(logits3, 1.0, check_rng) for _ in range(4000)])
freq = np.bincount(draws, minlength=3) / len(draws)
assert np.allclose(freq, [0.6, 0.3, 0.1], atol=0.03), f"at tau = 1 the draws should follow the probabilities, got {np.round(freq, 3)}"
cold = np.array([sample_next(logits3, 0.05, check_rng) for _ in range(200)])
assert (cold == 0).all(), "at tau = 0.05 the top token should win every draw"
p_hot = softmax(logits3 / 3.0)
hot = np.array([sample_next(logits3, 3.0, check_rng) for _ in range(4000)])
freq_hot = np.bincount(hot, minlength=3) / len(hot)
assert np.allclose(freq_hot, p_hot, atol=0.03), f"at tau = 3 the draws should follow softmax(logits / 3) = {np.round(p_hot, 3)}, got {np.round(freq_hot, 3)}"
assert p_hot[0] < 0.6 and p_hot[2] > 0.1, "a high temperature flattens the probabilities"

def generate_sampled(prompt_ids, W_out, b_out, tau, rng):
    ids = list(prompt_ids)
    while len(ids) < T and ids[-1] != END:
        H, _ = causal_self_attention(embed(np.array(ids)))
        ids.append(sample_next(H[-1] @ W_out + b_out, tau, rng))
    return np.array(ids)

text_rng = np.random.default_rng(1)
for tau in (0.3, 1.0, 3.0):
    print(f"tau = {tau}")
    for _ in range(3):
        sample = generate_sampled(np.array([START]), W, b, tau, text_rng)
        assert sample[0] == START and len(sample) <= T and (sample[-1] == END or len(sample) == T)
        print("   ", to_words(sample[1:]))
print("Part 6 passed")

## Solution

### Part 1: The probability of a sentence

In [ ]:
def score_sentence(conditionals):
    """(probability of the whole sentence, mean loss per token, perplexity)."""
    conditionals = np.asarray(conditionals, dtype=float)
    probability = conditionals.prod()
    loss = -np.log(conditionals).mean()
    return probability, loss, np.exp(loss)

### Part 2: Causal self-attention

In [ ]:
def causal_self_attention(X):
    """One layer of attention where each position sees only itself and earlier positions.
    Returns (H, weights): H is X plus what each position gathered, weights is the masked attention matrix."""
    n = len(X)
    Q, K, V = X @ W_q, X @ W_k, X @ W_v
    scores = Q @ K.T / np.sqrt(d)
    allowed = np.tril(np.ones((n, n), dtype=bool))
    scores = np.where(allowed, scores, -np.inf)
    weights = softmax(scores, axis=1)
    return X + weights @ V, weights

### Part 3: Shifted targets and the loss

In [ ]:
def lm_loss(tokens, W_out, b_out):
    """Mean next-token cross-entropy of one sequence: the model reads tokens[:-1] and is scored on tokens[1:]."""
    inputs, targets = tokens[:-1], tokens[1:]
    H, _ = causal_self_attention(embed(inputs))
    logits = H @ W_out + b_out
    m = logits.max(axis=1, keepdims=True)
    logp = logits - m - np.log(np.exp(logits - m).sum(axis=1, keepdims=True))
    return -logp[np.arange(len(targets)), targets].mean()

### Part 4: Train the output layer

In [ ]:
def head_gradient(tokens, W_out, b_out):
    """Gradient of lm_loss(tokens, W_out, b_out) with respect to W_out and b_out."""
    inputs, targets = tokens[:-1], tokens[1:]
    H, _ = causal_self_attention(embed(inputs))
    dlogits = softmax(H @ W_out + b_out, axis=1)
    dlogits[np.arange(len(targets)), targets] -= 1
    dlogits /= len(targets)
    return H.T @ dlogits, dlogits.sum(axis=0)

### Part 5: Greedy generation

In [ ]:
def generate_greedy(prompt_ids, W_out, b_out):
    """Extend prompt_ids one most-likely token at a time until <e> or T tokens. Returns all the ids."""
    ids = list(prompt_ids)
    while len(ids) < T and ids[-1] != END:
        H, _ = causal_self_attention(embed(np.array(ids)))
        logits = H[-1] @ W_out + b_out
        ids.append(int(np.argmax(logits)))
    return np.array(ids)

### Part 6: Sampling and temperature

In [ ]:
def sample_next(logits, tau, rng):
    """One id drawn from softmax(logits / tau)."""
    return int(rng.choice(len(logits), p=softmax(logits / tau)))